# Week 4 — Hallucination Pilot: African vs. Popularity-Matched Control Entities

Quick feasibility check before committing to the full Week 4 research proposal (themes #6/#7: hallucination and calibration on underrepresented African topics). 44 factual questions (`03_Experiments/pilot_questions.json`), 22 about African institutions/figures/events, 22 popularity/category-matched non-African controls on the *same question type* (e.g. "first president after independence" for both groups) — controls for the known "hallucination correlates with entity rarity" confound, not just "Africa vs. everything." Every question was verified against a cited source (`source_url` field) before being added, not taken from memory alone.

Tests recent open models: **Gemma 4 (E4B-it, released April 2026)** and Qwen2.5, at instruction-tuned checkpoints small enough for a single T4. **You verify the answers yourself** — no external annotators, matching the contract's "avoid complex human-data collection" guidance.

**Before running:** Settings → Accelerator → GPU T4 x2.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Pull the repo and install dependencies

Reuses `src.evaluate.generate_samples` and `src.utils.get_device` — no new evaluation code needed for a pilot this small. `HF_TOKEN` needed both for rate limits and because Gemma checkpoints require accepting Google's license on Hugging Face first (do that once, in a browser, before running this cell).

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone --filter=blob:none --no-checkout {repo_url}
%cd llm-alignment-internship
!git sparse-checkout init --cone
!git sparse-checkout set src 03_Experiments
!git checkout main
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate huggingface_hub

## 2. Confirm the GPU is visible, load the pilot questions

In [ ]:
import json
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

with open("03_Experiments/pilot_questions.json") as f:
    questions = json.load(f)
print(f"Loaded {len(questions)} questions ({sum(q['region']=='africa' for q in questions)} africa / {sum(q['region']=='control' for q in questions)} control)")

## 3. Test each model, one at a time

Sequential loading (load → generate all 20 answers → free memory → next model) — same memory-safe pattern as the Week 3 notebooks, so two ~2-3B models never need to be resident simultaneously. `do_sample=False` (greedy) for reproducibility. Add/remove entries in `MODELS` freely — Gemma checkpoints need you to have clicked "Acknowledge license" on their Hugging Face page at least once first.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.evaluate import generate_samples
from src.utils import get_device

MODELS = [
    "google/gemma-4-E4B-it",
    "Qwen/Qwen2.5-3B-Instruct",
]

all_results = {}

for model_name in MODELS:
    print(f"\n{'#'*80}\n# {model_name}\n{'#'*80}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
    model.to(get_device())

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": q["question"] + " Answer concisely."}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for q in questions
    ]
    with torch.no_grad():
        completions = generate_samples(model, tokenizer, prompts, max_new_tokens=60, do_sample=False)

    model_results = []
    for q, completion in zip(questions, completions):
        model_results.append({**q, "model_answer": completion.strip()})
        print("-" * 80)
        print(f"[{q['region']}/{q['category']}] {q['question']}")
        print(f"  ground truth : {q['ground_truth']}")
        print(f"  model answer : {completion.strip()}")

    all_results[model_name] = model_results

    del model
    torch.cuda.empty_cache()

with open("pilot_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved to pilot_results.json")

## 4. Next step

Read through the printed output (or `pilot_results.json`) and judge each answer yourself: correct / wrong-and-confident (hallucination) / vague-hedge (not a hallucination, per the TruthfulQA-style distinction). If African-entity questions show a clearly higher wrong-and-confident rate than their popularity-matched controls, that's the green light to write up the full Week 4 research proposal around this theme — with this pilot as the "initial experiment under 12 hours" the contract asks for.